6 
Ping pong básico: Crea un agente que controle una barra para devolver la pelota. Explica cómo aprende la trayectoria de la pelota y mejora sus respuestas.

Ejemplo simple de ping pong:
- su taamño es de width 6 y height de 5
- Hay una bola que se mueve hacia la derecha.
- Hay una barra que puede subir, quedarse quieta o bajar.
- Si la barra llega a la bola, el agente gana +1.
- Si la falla, pierde -1 y el juego termina.

El agente aprende probando acciones y guardando qué acción es mejor en cada situación.

In [36]:
import numpy as np

class PongEnv:
    def __init__(self, width=6, height=5, paddle_size=2):
        self.width = width
        self.height = height
        self.paddle_size = paddle_size
        self.actions = [-1, 0, 1]  # subir, quedarse quieto, bajar
        self.reset()

    def reset(self):
        self.ball_x = 0
        self.ball_y = self.height // 2
        self.vel_x = 1
        self.paddle_y = self.height // 2 - self.paddle_size // 2
        self.done = False
        return self._get_state()

    def _get_state(self):
        return (self.ball_x, self.ball_y, self.paddle_y)

    def step(self, action):
        if self.done:
            raise ValueError("El episodio ya terminó")

        self.paddle_y = np.clip(self.paddle_y + action, 0, self.height - self.paddle_size)
        self.ball_x += self.vel_x
        self.ball_y += self.vel_x

        if self.ball_y < 0:
            self.ball_y = 0
        if self.ball_y >= self.height:
            self.ball_y = self.height - 1

        reward = 0
        self.done = False

        if self.ball_x >= self.width - 1:
            if self.paddle_y <= self.ball_y < self.paddle_y + self.paddle_size:
                reward = 1
            else:
                reward = -1
            self.done = True

        return self._get_state(), reward, self.done

  

In [37]:
class Agent:
    def __init__(self, actions, alpha=0.5, prob_exp=0.5):
        self.actions = actions
        self.value_function = {}  # tabla con pares estado -> valor
        self.alpha = alpha         # learning rate
        self.positions = []        # guardamos todas las posiciones de la partida
        self.prob_exp = prob_exp   # probabilidad de explorar

    def reset(self):
        self.positions = []

    def choose_action(self, state, explore=True):
        # exploración: acción aleatoria
        if explore and np.random.uniform(0, 1) < self.prob_exp:
            return np.random.choice(self.actions)
        
        # explotación: acción con mayor valor esperado
        max_value = -1000
        best_action = self.actions[0]
        for action in self.actions:
            # aproximar el siguiente estado
            next_ball_x = state[0] + 1
            next_ball_y = state[1] + 1
            next_paddle_y = np.clip(state[2] + action, 0, 4)  # altura máxima = 5-1
            next_state = (next_ball_x, next_ball_y, next_paddle_y)
            next_state_str = str(next_state)
            value = self.value_function.get(next_state_str, 0.0)
            if value > max_value:
                max_value = value
                best_action = action
        return best_action

    def update(self, state):
        # guardar el estado visitado
        self.positions.append(str(state))

    def reward(self, reward):
        # al final de la partida, actualizar la tabla de valores
        # iteramos en reverso desde el final del episodio
        for p in reversed(self.positions):
            if self.value_function.get(p) is None:
                self.value_function[p] = 0.0
            self.value_function[p] += self.alpha * (reward - self.value_function[p])
            reward = self.value_function[p]

In [43]:
env = PongEnv()
# lo que le stamos mando actions es [-1, 0, 1] es el movimiento 
agent = Agent(actions=env.actions, alpha=0.5, prob_exp=0.7)

n_episodes = 4000
for episode in range(1, n_episodes + 1):
    state = env.reset()
    agent.reset()
    total_reward = 0
    done = False
    while not done:
        action = agent.choose_action(state, explore=True)
        next_state, reward, done = env.step(action)
        agent.update(next_state)
        state = next_state
        total_reward += reward
    agent.reward(total_reward)
    if episode % 200 == 0:
        print(f"Episodio {episode}: recompensa del último episodio = {total_reward}")
    

Episodio 200: recompensa del último episodio = -1
Episodio 400: recompensa del último episodio = -1
Episodio 600: recompensa del último episodio = -1
Episodio 800: recompensa del último episodio = -1
Episodio 1000: recompensa del último episodio = -1
Episodio 1200: recompensa del último episodio = -1
Episodio 1400: recompensa del último episodio = -1
Episodio 1600: recompensa del último episodio = -1
Episodio 1800: recompensa del último episodio = -1
Episodio 2000: recompensa del último episodio = -1
Episodio 2200: recompensa del último episodio = -1
Episodio 2400: recompensa del último episodio = -1
Episodio 2600: recompensa del último episodio = -1
Episodio 2800: recompensa del último episodio = -1
Episodio 3000: recompensa del último episodio = -1
Episodio 3200: recompensa del último episodio = -1
Episodio 3400: recompensa del último episodio = -1
Episodio 3600: recompensa del último episodio = -1
Episodio 3800: recompensa del último episodio = -1
Episodio 4000: recompensa del últim

Cómo funciona el aprendizaje:

1. El agente ve la posición de la bola y de la barra.
2. Elige subir, bajar o quedarse quieto.
3. Si devuelve la bola, recibe +1.
4. Si falla, recibe -1 y el juego termina.
5. Aprende guardando en una tabla qué acción es mejor en cada situación.

Poco a poco, el agente usa esa tabla para decidir mejor y devolver más bolas.